# Run All


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
_start = Path(os.getenv("PROJECT_ROOT") or Path.cwd()).resolve()
root = next(p for p in [_start, *_start.parents] if (p / "config.yaml").exists())
sys.path[:0] = [str(root), str(root / "src")]

import glob
import subprocess

import nbformat

repo_root = str(root)
source_folder = os.path.join(repo_root, "notebooks")

from src.utils.helpers import p, t, c

t("Run All Notebooks")

notebooks = glob.glob(os.path.join(source_folder, "*.ipynb"))
notebooks = [nb for nb in notebooks if os.path.basename(nb) in
             ["14_master_execution_plan_512_all.ipynb", "15_model_tracker_submission_manager.ipynb"]
             ]
notebooks = sorted(notebooks)

for nb in notebooks:
    print(os.path.basename(nb))


In [ ]:
import os
from nbclient import NotebookClient
from nbclient.exceptions import CellTimeoutError

failed_notebooks = []

# Set per cell timeout in seconds. Use None for no timeout.
#PER_CELL_TIMEOUT = 3600  # one hour per cell
PER_CELL_TIMEOUT = None

for nb_path in notebooks:
    nb_name = os.path.basename(nb_path)
    try:
        p()
        t(f"Executing: {nb_name}")

        nb = nbformat.read(nb_path, as_version = 4)
        client = NotebookClient(
                nb,
                timeout = PER_CELL_TIMEOUT,
                kernel_name = "python3",
        )
        client.execute()

        # Save executed notebook
        nbformat.write(nb, nb_path)
        p(f"Finished & saved: {nb_name}", color1 = c.BLUE)

        # Path relative to repo root
        rel_nb_path = os.path.relpath(nb_path, repo_root)

        # Git add/commit/push from repo root
        subprocess.run(
                ["git", "add", rel_nb_path],
                check = True,
                cwd = repo_root,
        )
        commit_msg = f"Auto-update {nb_name}"
        subprocess.run(
                ["git", "commit", "-m", commit_msg],
                check = False,
                cwd = repo_root,
        )
        subprocess.run(
                ["git", "push"],
                check = True,
                cwd = repo_root,
        )

        p(f"Committed and pushed: {nb_name}", color1 = c.GREEN)

    except CellTimeoutError as e:
        p(
                f"Timeout while executing {nb_name}",
                str(e),
                color1 = c.RED,
                color2 = c.BLACK,
        )
        failed_notebooks.append(nb_path)

    except Exception as e:
        p(f"Error while executing {nb_name}", str(e), color1 = c.RED, color2 = c.BLACK)
        failed_notebooks.append(nb_path)

t("Execution completed.")

if failed_notebooks:
    p("\n\nNotebooks that failed:", color1 = c.MAGENTA)
    for nb_path in failed_notebooks:
        p("", nb_path)
